# Give Claude a wallet: paying for resources with x402

Most APIs an agent might want to use sit behind an account, an API key, and a billing relationship a human set up in advance. [x402](https://x402.org) removes that step. A server answers an unauthenticated request with HTTP `402 Payment Required` and a machine-readable price; the client signs a stablecoin payment for exactly that amount, retries, and receives the resource plus an on-chain receipt. No signup, no invoice, no key to provision: an agent holding a wallet can buy what it needs, one request at a time.

This cookbook builds a Claude agent that does exactly that, safely:

1. It reads a `402` response and the payment terms inside it before spending anything.
2. It holds a wallet behind two independent spend limits: one enforced by the x402 SDK before any signature is produced, one enforced by your own code across the whole session.
3. It uses three tools (`look_at_wall`, `quote`, `buy_block`) with the Anthropic SDK's tool runner to find something worth buying, pay for it, and verify the purchase on-chain.

The resource we buy is a block on [The 402 Wall](https://404humans.xyz): a 1600×900 pixel billboard split into 14,400 blocks of 10×10 pixels, sold only over x402 for $1.00 USDC on Base, and only to machines. It is a good teaching target because a purchase has a visible, verifiable side effect: the block appears on the wall and the payment appears on Basescan. The same client code works against any x402 resource.

**Disclosure.** The notebook author operates The 402 Wall. It is used here as a public demo endpoint, not as a required dependency of x402 or this client.

**What this costs to run.** One block is $1.00 USDC (the facilitator pays the gas, so the wallet needs no ETH), plus a few cents of Claude usage. If `EVM_PRIVATE_KEY` is not set, every cell still runs and the purchase step is skipped.

## Step 1: Set up the environment

Install the Anthropic SDK and the x402 SDK with its EVM and `requests` extras. Create a `.env` file in the same directory as this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_MODEL=claude-opus-5   # optional
EVM_PRIVATE_KEY=0x...   # optional: a wallet holding a little USDC on Base
```

Use a dedicated wallet for agents, funded with only what you intend them to spend. Never point an agent at a key that holds more than you would hand to a stranger.

In [1]:
%pip install "anthropic>=1.0" "x402[evm,requests]>=2.21" python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import base64
import json
import os
from dataclasses import dataclass, field

import anthropic
import requests
from anthropic import beta_tool
from dotenv import load_dotenv
from eth_account import Account
from x402 import AbortResult, x402ClientSync
from x402.http.clients import x402_requests
from x402.http.utils import decode_payment_required_header, decode_payment_response_header
from x402.mechanisms.evm.exact import ExactEvmScheme
from x402.mechanisms.evm.signers import EthAccountSigner

load_dotenv()

client = anthropic.Anthropic()
MODEL_NAME = os.environ.get("ANTHROPIC_MODEL", "claude-opus-5")

WALL = "https://404humans.xyz"
NETWORK = "eip155:8453"  # Base mainnet, as a CAIP-2 chain id
USDC = "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913"
HTTP_TIMEOUT = 60  # a paid request includes on-chain settlement; give it time

## Step 2: Read a 402 before paying for anything

Ask for an unclaimed block with a plain, unauthenticated request. The server refuses with `402` and puts its terms in a `PAYMENT-REQUIRED` header: base64-encoded JSON that the x402 SDK decodes into a typed object. Nothing has been signed or spent yet. This is a price tag.

In [3]:
probe = requests.get(f"{WALL}/blocks/80,45", timeout=HTTP_TIMEOUT)
print("HTTP", probe.status_code)

if probe.status_code != 402:
    print("That block has been sold since this notebook was written:", probe.json())
else:
    terms = decode_payment_required_header(probe.headers["PAYMENT-REQUIRED"])
    offer = terms.accepts[0]  # a server may list several ways to pay; this one lists one

    print("x402 version :", terms.x402_version)
    print("resource     :", terms.resource.url)
    print("description  :", terms.resource.description)
    print("scheme       :", offer.scheme)
    print("network      :", offer.network)
    print("amount       :", offer.amount, "atomic units (USDC has 6 decimals)")
    print("asset        :", offer.asset)
    print("pay to       :", offer.pay_to)
    print("valid for    :", offer.max_timeout_seconds, "seconds")

HTTP 402
x402 version : 2
resource     : https://404humans.xyz/blocks/80,45
description  : One 10x10 pixel block on The 402 Wall (404humans.xyz). $1.00 in USDC buys 100 pixels, a permanent deed, and a line in the Founding Machines registry. Optional query params on the paid request: name, framework, message, link, px (100 hex-encoded palette bytes).
scheme       : exact
network      : eip155:8453
amount       : 1000000 atomic units (USDC has 6 decimals)
asset        : 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913
pay to       : 0x1f3f6191bea8a57dd7a1fe90a4249c706ad4fb46
valid for    : 300 seconds


## Step 3: Give the agent a wallet, with two spend limits

An x402 payment on an EVM chain is an [EIP-3009](https://eips.ethereum.org/EIPS/eip-3009) `transferWithAuthorization`: the wallet signs a message authorizing a transfer of an exact amount to an exact recipient, valid for a short window. The facilitator submits it on-chain and pays the gas. The agent's wallet therefore only needs USDC, and only needs to sign; it never holds ETH or sends transactions itself.

Because signing *is* spending, put the limits in front of the signature:

- **Per-payment cap and network allowlist, enforced by the SDK.** `x402ClientSync` evaluates `spend_controls` while choosing which offer to accept, and only the Base payment scheme is registered. An offer above the cap, for an unsupported asset, or on another network is rejected before any signature exists.
- **Session budget, enforced by your code.** The SDK caps each payment but does not track cumulative spend. A small `Budget` object does, and the `buy_block` tool consults it before every purchase. Here the budget is exactly one block's price, so however the model behaves, a second purchase cannot happen. If a signed request has an ambiguous outcome, the budget remains consumed until you reconcile it.

If `EVM_PRIVATE_KEY` is not set, the notebook keeps going in read-only mode.

In [4]:
payment_state = {"authorization_created": False, "expected_terms": None}


def require_expected_terms(context):
    """Abort if the offer selected for signing differs from the quote we approved."""
    expected = payment_state["expected_terms"]
    selected = context.selected_requirements
    if expected is None or (
        int(selected.amount) != expected["amount_atomic"]
        or selected.asset.lower() != expected["asset"].lower()
        or selected.network != expected["network"]
        or selected.pay_to.lower() != expected["pay_to"].lower()
    ):
        return AbortResult("selected payment terms differ from the approved quote")
    return None


def note_authorization_created(_context) -> None:
    """Remember that a signed authorization may already be settleable."""
    payment_state["authorization_created"] = True


private_key = os.environ.get("EVM_PRIVATE_KEY")
account = Account.from_key(private_key) if private_key else None

if account is None:
    pay_session = None
    print("No EVM_PRIVATE_KEY set: running read-only, purchases will be skipped.")
else:
    x402_client = (
        x402ClientSync()
        # Registering only Base makes the supported network list an allowlist.
        .register(NETWORK, ExactEvmScheme(signer=EthAccountSigner(account)))
        .set_spend_controls({"max_amount_per_payment": "$1"})  # refuse to sign anything larger
        .on_before_payment_creation(require_expected_terms)
        .on_after_payment_creation(note_authorization_created)
    )
    # A requests.Session that answers 402s on its own: sign, retry, return the paid response.
    pay_session = x402_requests(x402_client)
    print("Agent wallet:", account.address)


@dataclass
class Budget:
    """Cumulative spend limit for one agent session, in atomic USDC units."""

    limit: int
    spent: int = 0
    purchases: list[dict] = field(default_factory=list)

    def can_afford(self, amount: int) -> bool:
        return self.spent + amount <= self.limit

    def record(self, coord: str, amount: int, tx: str) -> None:
        self.spent += amount
        self.purchases.append({"coord": coord, "atomic": amount, "tx": tx})


# One block costs 1.00 USD, so this budget makes a second purchase impossible by construction.
budget = Budget(limit=1_000_000)

Agent wallet: 0x689861913fcfdDEA5EF7Da824047b5dc55862198


## Step 4: Define the tools

Three tools, each doing one thing, with descriptions written for the model rather than for us. The description is what Claude uses to decide when a tool is appropriate.

- `look_at_wall` is free. It returns the wall's public state: size, price, what is already claimed, and the color legend used to draw a block.
- `quote` is free. It returns one block's status and, if the block is for sale, records the exact terms from its `402`.
- `buy_block` is paid. It requires a prior quote, re-reads the terms, rejects any change, checks the session budget, then makes one request through the payment-aware session. On success it decodes the `PAYMENT-RESPONSE` receipt, which carries the settlement transaction hash. Any ambiguous outcome after a signature has been created is treated as *possibly paid*: it counts against the budget, so the agent cannot pay twice while the outcome is unknown.

Each tool prints a one-line trace so the run is easy to follow. Tools return JSON strings, which Claude reads as text.

In [5]:
# Characters the agent can draw with, mapped to indices in the wall's palette.
LEGEND = {
    ".": 1,  # black
    "W": 8,  # white
    "K": 3,  # dark gray
    "L": 6,  # light gray
    "B": 18,  # blue
    "C": 14,  # cyan
    "G": 10,  # green
    "Y": 30,  # yellow
    "O": 27,  # orange
    "R": 26,  # red
    "M": 23,  # magenta
    "P": 22,  # purple
}


def trace(direction: str, text: str) -> None:
    print(f"  {direction} {text}")


def usd(atomic_amount: str) -> float:
    """USDC has 6 decimals: 1000000 atomic units is 1.00 USD."""
    return int(atomic_amount) / 1_000_000


def valid_coord(coord: str) -> bool:
    """Accept only canonical wall coordinates, never an arbitrary URL path."""
    try:
        col_text, row_text = coord.split(",")
        col, row = int(col_text), int(row_text)
    except (ValueError, TypeError):
        return False
    return coord == f"{col},{row}" and 0 <= col < 160 and 0 <= row < 90


def terms_for(coord: str) -> dict:
    """Free read of one block: its deed if sold, its 402 terms if for sale."""
    r = requests.get(f"{WALL}/blocks/{coord}", timeout=HTTP_TIMEOUT)
    if r.status_code == 402:
        terms = decode_payment_required_header(r.headers["PAYMENT-REQUIRED"])
        offer = next(
            (
                candidate
                for candidate in terms.accepts
                if candidate.scheme == "exact"
                and candidate.network == NETWORK
                and candidate.asset.lower() == USDC.lower()
            ),
            None,
        )
        if offer is None:
            return {"status": "unsupported_payment_terms", "coord": coord}
        return {
            "status": "for_sale",
            "coord": coord,
            "amount_atomic": int(offer.amount),
            "price_usd": usd(offer.amount),
            "asset": offer.asset,
            "network": offer.network,
            "pay_to": offer.pay_to,
            "description": terms.resource.description,
        }
    if r.status_code == 200:
        return {"status": "sold", "coord": coord, "deed": r.json()}
    return {"status": "error", "coord": coord, "http": r.status_code, "body": r.text[:300]}


def describe_failure(response: requests.Response) -> str:
    """Turn a failed paid request into one line the agent can act on.

    A declined payment comes back as a fresh 402 whose PAYMENT-REQUIRED header carries the
    reason; the wall's other failures (block taken, settlement unknown) explain themselves in
    a JSON body.
    """
    if response.status_code == 402 and "PAYMENT-REQUIRED" in response.headers:
        terms = decode_payment_required_header(response.headers["PAYMENT-REQUIRED"])
        return f"payment declined by the facilitator: {terms.error}"
    try:
        detail = response.json().get("message") or ""
    except ValueError:
        detail = ""
    return f"HTTP {response.status_code}: {detail}" if detail else f"HTTP {response.status_code}"


quoted_terms = {}


@beta_tool
def look_at_wall() -> str:
    """Get the public state of The 402 Wall: grid size, price per block, how many blocks are
    claimed, the coordinates already taken, and the drawing legend that buy_block accepts.

    Coordinates are "col,row" with col 0-159 and row 0-89. Costs nothing.
    """
    trace("->", "look_at_wall()")
    wall = requests.get(f"{WALL}/api/canvas", timeout=HTTP_TIMEOUT).json()
    cols = wall["canvas"]["cols"]
    claimed = [f"{b['i'] % cols},{b['i'] // cols}" for b in wall["blocks"]]
    result = {
        "grid": {"cols": cols, "rows": wall["canvas"]["rows"], "block_pixels": 10},
        "pricing": wall["pricing"],
        "stats": wall["stats"],
        "claimed_coords": claimed,
        "legend": LEGEND,
    }
    trace("<-", f"{wall['stats']['claimed']} claimed, {wall['stats']['remaining']} for sale")
    return json.dumps(result)


@beta_tool
def quote(coord: str) -> str:
    """Check one block before buying: whether it is for sale and, if so, the exact x402 terms
    (price in USD, asset, network, recipient). Costs nothing. Always quote before buy_block.

    Args:
        coord: Block coordinate as "col,row", for example "12,34".
    """
    trace("->", f"quote({coord})")
    if not valid_coord(coord):
        result = {"status": "invalid_coordinate", "coord": coord}
        trace("<-", result["status"])
        return json.dumps(result)
    result = terms_for(coord)
    if result["status"] == "for_sale":
        quoted_terms[coord] = result
    detail = f" at {result['price_usd']:.2f} USD" if result["status"] == "for_sale" else ""
    trace("<-", f"{result['status']}{detail}")
    return json.dumps(result)


@beta_tool
def buy_block(coord: str, name: str, message: str, link: str, pixel_rows: list[str]) -> str:
    """Buy one unclaimed block on The 402 Wall by paying its x402 price in USDC, and paint it.

    Re-reads the current terms, checks them against the session budget, then pays. Returns the
    settlement transaction hash on success, or the reason if the purchase was declined or failed.

    Args:
        coord: Block coordinate as "col,row". Quote it first.
        name: Display name for this agent in the wall's registry (1-40 characters).
        message: Short message shown when someone hovers the block (1-140 characters).
        link: An https URL the block links to.
        pixel_rows: The 10x10 artwork as exactly 10 strings of exactly 10 characters, top row
            first, using only characters from the legend returned by look_at_wall (for example
            "." for black and "B" for blue).
    """
    trace("->", f"buy_block({coord}, name={name!r})")
    if pay_session is None:
        return json.dumps({"ok": False, "reason": "no wallet configured; purchase skipped"})

    if not valid_coord(coord):
        return json.dumps({"ok": False, "reason": "invalid block coordinate"})
    quoted = quoted_terms.get(coord)
    if quoted is None:
        return json.dumps({"ok": False, "reason": "quote this block before buying it"})

    # Validate the artwork before spending anything.
    if len(pixel_rows) != 10 or any(len(row) != 10 for row in pixel_rows):
        return json.dumps({"ok": False, "reason": "pixel_rows must be 10 rows of 10 characters"})
    unknown = {ch for row in pixel_rows for ch in row} - set(LEGEND)
    if unknown:
        return json.dumps({"ok": False, "reason": f"unknown legend characters: {sorted(unknown)}"})
    px = "".join(f"{LEGEND[ch]:02x}" for row in pixel_rows for ch in row)

    # Re-read the price at purchase time and check the session budget.
    current = terms_for(coord)
    quoted_terms.pop(coord, None)  # a quote authorizes at most one payment attempt
    if current["status"] != "for_sale":
        return json.dumps({"ok": False, "reason": f"block is not for sale: {current['status']}"})
    if (
        current["amount_atomic"] != quoted["amount_atomic"]
        or current["asset"].lower() != quoted["asset"].lower()
        or current["network"] != quoted["network"]
        or current["pay_to"].lower() != quoted["pay_to"].lower()
    ):
        return json.dumps({"ok": False, "reason": "payment terms changed; quote again"})
    amount = current["amount_atomic"]
    price = current["price_usd"]
    if not budget.can_afford(amount):
        trace(
            "<-", f"declined: {price:.2f} USD would exceed the {usd(budget.limit):.2f} USD budget"
        )
        return json.dumps(
            {
                "ok": False,
                "reason": "over session budget",
                "price_usd": price,
                "spent_usd": usd(budget.spent),
            }
        )

    # One request. The session handles the 402: sign the exact amount, retry, return the result.
    payment_state["authorization_created"] = False
    payment_state["expected_terms"] = current
    try:
        paid = pay_session.get(
            f"{WALL}/blocks/{coord}",
            params={
                "name": name,
                "framework": "Claude tool runner",
                "message": message,
                "link": link,
                "px": px,
            },
            timeout=HTTP_TIMEOUT,
        )
    except Exception as e:  # the tool must return a result, not crash the loop
        result = {"ok": False, "reason": f"payment failed: {e}"}
        if payment_state["authorization_created"]:
            budget.record(coord, amount, tx="unknown")
            result["note"] = "outcome unknown; counted as spent. Do not pay again; reconcile later."
        trace("<-", f"failed: {e}")
        return json.dumps(result)
    finally:
        payment_state["expected_terms"] = None

    receipt_header = paid.headers.get("PAYMENT-RESPONSE")
    if not receipt_header:
        reason = describe_failure(paid)
        result = {"ok": False, "reason": reason, "http": paid.status_code}
        if payment_state["authorization_created"]:
            # Without a receipt, a signed authorization has an unknown outcome.
            budget.record(coord, amount, tx="unknown")
            result["note"] = "outcome unknown; counted as spent. Do not pay again; reconcile later."
        trace("<-", f"failed: {reason}")
        return json.dumps(result)

    try:
        receipt = decode_payment_response_header(receipt_header)
    except Exception as e:
        result = {"ok": False, "reason": f"invalid payment receipt: {e}"}
        if payment_state["authorization_created"]:
            budget.record(coord, amount, tx="unknown")
            result["note"] = "outcome unknown; counted as spent. Do not pay again; reconcile later."
        trace("<-", f"receipt could not be decoded: {e}")
        return json.dumps(result)

    if not receipt.success or receipt.network != NETWORK or not receipt.transaction:
        result = {"ok": False, "reason": "receipt did not prove a successful Base settlement"}
        if payment_state["authorization_created"]:
            budget.record(coord, amount, tx="unknown")
            result["note"] = "outcome unknown; counted as spent. Do not pay again; reconcile later."
        trace("<-", result["reason"])
        return json.dumps(result)

    budget.record(coord, amount, receipt.transaction)
    if paid.status_code != 200:
        reason = describe_failure(paid)
        trace("<-", f"paid, but the resource failed: {reason}")
        return json.dumps(
            {
                "ok": False,
                "paid": True,
                "reason": reason,
                "http": paid.status_code,
                "tx": receipt.transaction,
                "note": "payment settled; do not retry. Reconcile the resource separately.",
            }
        )
    trace("<-", f"settled: {price:.2f} USD, tx {receipt.transaction}")
    return json.dumps(
        {
            "ok": True,
            "coord": coord,
            "paid_usd": price,
            "network": receipt.network,
            "tx": receipt.transaction,
            "explorer": f"https://basescan.org/tx/{receipt.transaction}",
            "deed": paid.json(),
        }
    )

## Step 5: Run the agent

The tool runner drives the loop: it sends the tools and the task to Claude, executes whatever tool Claude calls, feeds the result back, and stops when Claude answers without calling a tool. We print Claude's own text between tool calls so its reasoning sits next to the trace.

The system prompt gives the agent a goal, a budget, and one rule: quote before buying. Claude decides which block to buy and what to draw.

In [6]:
SYSTEM_PROMPT = f"""You are an autonomous purchasing agent with a small USDC wallet.
Your session budget is {usd(budget.limit):.2f} USD. You may spend it only on The 402 Wall, and only on
blocks you have quoted first. Never buy more than one block.

Task: buy exactly one unclaimed block and paint it. Look at the wall, choose a block that is not
already claimed and not directly adjacent to one, quote it, and if it costs at most 1.00 USD, buy it.
Design a simple, recognizable 10x10 glyph (a letter, a heart, an arrow) using the legend.
Register as "x402_paying_agent" with the message "Bought by a Claude agent running the x402
cookbook recipe" and the link https://github.com/anthropics/claude-cookbooks.

When you are done, report what you bought, what it cost, and the transaction hash. If you could
not buy, report exactly why.
"""

runner = client.beta.messages.tool_runner(
    model=MODEL_NAME,
    max_tokens=16000,
    system=SYSTEM_PROMPT,
    tools=[look_at_wall, quote, buy_block],
    messages=[{"role": "user", "content": "Go ahead."}],
    max_iterations=12,  # hard stop for an autonomous loop that can spend money
)

for message in runner:
    for block in message.content:
        if block.type == "text" and block.text.strip():
            print(f"\nClaude: {block.text.strip()}\n")


Claude: I'll start by looking at the wall to see what's available.

  -> look_at_wall()


  <- 3 claimed, 14397 for sale



Claude: Three blocks claimed. Let me quote a spot far from all of them.

  -> quote(80,45)


  <- for_sale at 1.00 USD



Claude: Price is exactly $1.00, within budget. Buying it with a heart glyph.

  -> buy_block(80,45, name='x402_paying_agent')


  <- settled: 1.00 USD, tx 0x3dd64c4b7d2bc893c6cffd1c19cdb8285e29460fae3a1977ecc9b9cf0526b35c



Claude: **Purchase complete.**

**What I bought:** Block `80,45` on The 402 Wall (160×90 grid, block index 7280). The wall only had 3 blocks claimed (`95,38`, `58,54`, `75,60`), so I picked a location well clear of all of them — no adjacency issues.

**What it cost:** $1.00 USDC on Base (eip155:8453) — exactly the per-block price and exactly at my session budget cap. I quoted it first and confirmed the terms (price, asset, network, recipient `0x1f3f…fb46`) before paying.

**Transaction hash:**
`0x3dd64c4b7d2bc893c6cffd1c19cdb8285e29460fae3a1977ecc9b9cf0526b35c`
[View on Basescan](https://basescan.org/tx/0x3dd64c4b7d2bc893c6cffd1c19cdb8285e29460fae3a1977ecc9b9cf0526b35c)

**The artwork:** A red heart on black, using `R` (red) and `.` (black) from the wall's legend:

```
..........
.RR....RR.
RRRR..RRRR
RRRRRRRRRR
RRRRRRRRRR
.RRRRRRRR.
..RRRRRR..
...RRRR...
....RR....
..........
```

**Registry entry:** name `x402_paying_agent`, message "Bought by a Claude agent running the x402 cookboo

## Step 6: Verify the purchase

Trust the receipt, but check. The block is now a public deed anyone can read without paying, the transaction is on Basescan, and the budget ledger shows what the session spent. The deed also returns the pixels, so we can decode the artwork back through the legend.

In [7]:
print(f"Session spend: {usd(budget.spent):.2f} / {usd(budget.limit):.2f} USD")

REVERSE_LEGEND = {index: char for char, index in LEGEND.items()}

for purchase in budget.purchases:
    check = requests.get(f"{WALL}/blocks/{purchase['coord']}", timeout=HTTP_TIMEOUT)
    print(f"\nBlock {purchase['coord']}, public deed (HTTP {check.status_code}):")
    if check.status_code != 200:
        print("Purchase is not yet verifiable. Do not retry payment; reconcile later.")
        continue
    deed = check.json()
    print(
        json.dumps(
            {k: deed.get(k) for k in ("block", "owner", "settled", "tx", "message", "link")},
            indent=2,
        )
    )
    if deed.get("pixels"):
        print("\nArtwork as stored on the wall:")
        pixels = base64.b64decode(deed["pixels"])
        for row in range(10):
            print(
                "   "
                + "".join(REVERSE_LEGEND.get(b, "?") for b in pixels[row * 10 : (row + 1) * 10])
            )
    tx = deed.get("tx") or purchase["tx"]
    print(f"\nExplorer: https://basescan.org/tx/{tx}")
    print(f"Wall:     {WALL}")

if not budget.purchases:
    print("No purchase was made in this session.")

Session spend: 1.00 / 1.00 USD



Block 80,45, public deed (HTTP 200):
{
  "block": {
    "coord": "80,45",
    "index": 7280
  },
  "owner": {
    "name": "x402_paying_agent",
    "wallet": "0x6898\u20262198",
    "framework": "Claude tool runner"
  },
  "settled": true,
  "tx": "0x3dd64c4b7d2bc893c6cffd1c19cdb8285e29460fae3a1977ecc9b9cf0526b35c",
  "message": "Bought by a Claude agent running the x402 cookbook recipe",
  "link": "https://github.com/anthropics/claude-cookbooks"
}

Artwork as stored on the wall:
   ..........
   .RR....RR.
   RRRR..RRRR
   RRRRRRRRRR
   RRRRRRRRRR
   .RRRRRRRR.
   ..RRRRRR..
   ...RRRR...
   ....RR....
   ..........

Explorer: https://basescan.org/tx/0x3dd64c4b7d2bc893c6cffd1c19cdb8285e29460fae3a1977ecc9b9cf0526b35c
Wall:     https://404humans.xyz


## What to take from this

**Read the 402 first.** The `quote` tool exists so the agent never pays a price it has not seen. Treat a `402` as data: decode it, put the numbers in front of the model (and, when it matters, a human), and only then let the paying client near it.

**Layer the limits.** The SDK's `spend_controls` and policies guard every individual signature; the `Budget` guards the session. Neither depends on the model behaving. A production agent adds a third layer at the wallet itself: fund it with the budget and nothing more.

**An ambiguous outcome after signing is not a failure.** An `exact` authorization names a specific amount, recipient, and validity window, and can settle at most once; but a fresh call through the paying session produces a fresh authorization, which is a second payment if the first one did settle. The SDK hook records whether an authorization was created, so `buy_block` consumes the budget after a timeout, malformed receipt, or unsuccessful response only when signing actually occurred. Reconcile afterwards by reading the resource (here, quoting the block), or use the [`payment-identifier`](https://github.com/x402-foundation/x402) extension, which lets a server recognize a retried payment as the same one.

**Human in the loop is one function away.** `buy_block` already re-reads the terms before paying. To require approval, gate on the quote:

```python
def approve(quote: dict) -> bool:
    prompt = f"Pay {quote['price_usd']:.2f} USD to {quote['pay_to']} for block {quote['coord']}? [y/N] "
    return input(prompt).strip().lower() == "y"
```

and call it before `pay_session.get(...)`. The tool runner does not need to change.

**Where to go next.** The x402 client used here is transport-agnostic: the same `x402ClientSync` wraps `httpx`, powers paid MCP tool calls (`x402.mcp`), and can discover paid resources through the Bazaar extension. To experiment without real funds, run one of the [x402 example servers](https://github.com/x402-foundation/x402/tree/main/examples) on Base Sepolia (`eip155:84532`) with the public facilitator at `https://x402.org/facilitator`, and point this client at it.